## Import danych

In [57]:
import kagglehub
import pandas as pd
from kagglehub import KaggleDatasetAdapter

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "syedanwarafridi/vehicle-sales-data",
    "car_prices.csv",
)

# Wyswietla Data Frame
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 558837 entries, 0 to 558836
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   year          558837 non-null  int64  
 1   make          548536 non-null  str    
 2   model         548438 non-null  str    
 3   trim          548186 non-null  str    
 4   body          545642 non-null  str    
 5   transmission  493485 non-null  str    
 6   vin           558833 non-null  str    
 7   state         558837 non-null  str    
 8   condition     547017 non-null  float64
 9   odometer      558743 non-null  float64
 10  color         558088 non-null  str    
 11  interior      558088 non-null  str    
 12  seller        558837 non-null  str    
 13  mmr           558799 non-null  float64
 14  sellingprice  558825 non-null  float64
 15  saledate      558825 non-null  str    
dtypes: float64(4), int64(1), str(11)
memory usage: 68.2 MB


### Usuwamy wiersze ktore nie zawieraja pelnych danych i kolumny ktore nie beda przydatne.

In [58]:
df = df.dropna(
    subset=["sellingprice", "year", "make", "saledate", "transmission", "body", "condition", "color"]
)

df = df.drop(columns=["vin", "state", "saledate", "seller"])

### Przetwarzamy wartosci i usuwamy bledne wpisy.

In [59]:
columns = ["sellingprice", "odometer", "condition"]
for x in columns:
    df[x] = pd.to_numeric(df[x], errors="coerce")


df.info()


<class 'pandas.DataFrame'>
Index: 472434 entries, 0 to 558836
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   year          472434 non-null  int64  
 1   make          472434 non-null  str    
 2   model         472346 non-null  str    
 3   trim          472434 non-null  str    
 4   body          472434 non-null  str    
 5   transmission  472434 non-null  str    
 6   condition     472434 non-null  float64
 7   odometer      472413 non-null  float64
 8   color         472434 non-null  str    
 9   interior      472434 non-null  str    
 10  mmr           472434 non-null  float64
 11  sellingprice  472434 non-null  float64
dtypes: float64(4), int64(1), str(7)
memory usage: 46.9 MB


### Grupujemy wiele szczegolowych naz nadwozia do maksymalnie 10 typow.

In [60]:
def normalize_body_type(body):
    body = str(body).lower().strip()

    if any(word in body for word in ["suv", "crossover", "sport utility"]):
        return "SUV / Crossover"
    if any(word in body for word in ["sedan", "saloon"]):
        return "Sedan"
    if any(word in body for word in ["hatchback", "hatch"]):
        return "Hatchback"
    if any(word in body for word in ["coupe", "koup"]):
        return "Coupe"
    if any(word in body for word in ["convertible", "cabri", "roadster"]):
        return "Convertible"
    if any(word in body for word in ["wagon", "estate"]):
        return "Wagon"
    if any(word in body for word in ["minivan", "passenger van"]):
        return "Minivan"
    if "van" in body:
        return "Van"
    if any(
        word in body
        for word in [
            "cab",
            "pickup",
            "truck",
            "crew",
            "supercrew",
            "regular",
            "extended",
            "quad",
            "king",
            "access",
        ]
    ):
        return "Pickup"

    return "Other"

df["body_type"] = df["body"].apply(normalize_body_type)
df["body_type"].value_counts()

body_type
Sedan              218336
SUV / Crossover    120970
Pickup              40499
Hatchback           23822
Minivan             21938
Coupe               18161
Wagon               14266
Convertible          9746
Van                  4696
Name: count, dtype: int64

### Top 5 najdrozszych samochodow.

In [61]:
df.nlargest(5, "sellingprice").sort_values("sellingprice", ascending=False)

,year,make,model,trim,body,transmission,condition,odometer,color,interior,mmr,sellingprice,body_type
344905,2014,Ford,Escape,Titanium,SUV,automatic,43.0,27802.0,green,tan,22800.0,230000.0,SUV / Crossover
548169,2011,Ferrari,458 Italia,Base,coupe,automatic,46.0,12116.0,red,black,182000.0,183000.0,Coupe
446949,2015,Mercedes-Benz,S-Class,S65 AMG,Sedan,automatic,41.0,5277.0,white,white,170000.0,173000.0,Sedan
545523,2013,Rolls-Royce,Ghost,Base,sedan,automatic,42.0,7852.0,white,beige,178000.0,171500.0,Sedan
125095,2012,Rolls-Royce,Ghost,Base,Sedan,automatic,45.0,14316.0,black,beige,154000.0,169500.0,Sedan


### Top 5 najtanszych samochodow.

In [62]:
df.nsmallest(5, "sellingprice").sort_values("sellingprice", ascending=False)

,year,make,model,trim,body,transmission,condition,odometer,color,interior,mmr,sellingprice,body_type
25588,2003,Chevrolet,Malibu,LS,Sedan,automatic,1.0,186893.0,red,beige,950.0,100.0,Sedan
196184,2004,Pontiac,Montana,Base,Minivan,automatic,2.0,106495.0,silver,gray,1700.0,100.0,Minivan
205309,2002,Chrysler,Sebring,GTC,Convertible,automatic,1.0,94937.0,black,tan,1325.0,100.0,Convertible
48453,2003,Mercedes-Benz,E-Class,E500,Sedan,automatic,21.0,1.0,black,black,7325.0,1.0,Sedan
293223,2014,Ford,E-Series Van,E-250,E-Series Van,automatic,41.0,31886.0,white,gray,20800.0,1.0,Van


In [63]:
import plotly.graph_objects as go

color_counts = (
    df["color"]
    .fillna("unknown")
    .astype(str)
    .str.lower()
    .str.strip()
    .replace({"": "unknown", "nan": "unknown", "none": "unknown", "null": "unknown"})
    .value_counts()
)

popular_colors = color_counts.nlargest(12)
other_colors = color_counts.iloc[12:].sum()
if other_colors > 0:
    popular_colors = pd.concat([popular_colors, pd.Series({"other": other_colors})])
popular_colors = popular_colors.astype(int)

car_color_map = {
    "black": "black", "white": "white", "silver": "silver",
    "gray": "gray", "grey": "gray", "blue": "royalblue",
    "red": "red", "green": "green", "brown": "saddlebrown",
    "gold": "gold", "yellow": "yellow", "orange": "orange",
    "purple": "purple", "beige": "beige", "burgundy": "#800020",
    "turquoise": "turquoise", "charcoal": "#36454F",
    "off-white": "#FAF9F6", "unknown": "gainsboro", "other": "lightgray",
}

fig = go.Figure(data=[go.Pie(
    labels=[("Unknown" if c == "unknown" else c.title()) for c in popular_colors.index],
    values=popular_colors.to_list(),
    marker=dict(
        colors=[car_color_map.get(c, "lightgray") for c in popular_colors.index],
        line=dict(color="white", width=2.5)
    ),
    hole=0.42,
    direction="clockwise",
    rotation=90,  # was startangle=90
    textposition="inside",
    texttemplate="%{percent:.1%}",
    hovertemplate="<b>%{label}</b><br>Count: %{value:,}<br>Share: %{percent:.1%}<extra></extra>",
)])

fig.update_layout(
    title=dict(text="Najpopularniejsze kolory samochodow", font=dict(size=16)),
    legend=dict(title="Kolor", orientation="v"),
    annotations=[dict(
        text=f"{popular_colors.sum():,}<br>samochodow",
        x=0.5, y=0.5,
        font=dict(size=13),
        showarrow=False
    )],
)

fig.show()



### Trenowanie i porownanie kilku modeli przewidywania ceny samochodu.

In [ ]:
import time

import numpy as np
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, SGDRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

model_df = df.copy()
model_df = model_df.dropna(subset=["sellingprice"])
model_df = model_df[model_df["sellingprice"] > 0]

train_index, test_index = train_test_split(
    model_df.index,
    test_size=0.2,
    random_state=42,
)
train_df = model_df.loc[train_index].copy()
test_df = model_df.loc[test_index].copy()

categorical_features = [
    column
    for column in [
        "make",
        "model",
        "trim",
        "body",
        "transmission",
        "color",
    ]
    if column in model_df.columns
]
base_numeric_features = [
    column for column in ["year", "condition", "odometer"] if column in model_df.columns
]
mmr_numeric_features = (
    base_numeric_features + ["mmr"]
    if "mmr" in model_df.columns
    else base_numeric_features
)


def build_price_model(model_type, numeric_features, categorical_features):
    if model_type in ["ridge", "sgd"]:
        numeric_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]
        )
        categorical_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        min_frequency=50,
                    ),
                ),
            ]
        )
        regressor = (
            Ridge(alpha=1.0)
            if model_type == "ridge"
            else SGDRegressor(
                loss="squared_error",
                penalty="l2",
                alpha=0.0001,
                max_iter=1000,
                tol=1e-3,
                random_state=42,
            )
        )
    else:
        numeric_transformer = Pipeline(
            steps=[("imputer", SimpleImputer(strategy="median"))]
        )
        categorical_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                (
                    "ordinal",
                    OrdinalEncoder(
                        handle_unknown="use_encoded_value",
                        unknown_value=-1,
                    ),
                ),
            ]
        )
        regressor = HistGradientBoostingRegressor(
            max_iter=120,
            learning_rate=0.08,
            l2_regularization=0.1,
            random_state=42,
        )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "regressor",
                TransformedTargetRegressor(
                    regressor=regressor,
                    func=np.log1p,
                    inverse_func=np.expm1,
                ),
            ),
        ]
    )


def estimate_matrix_memory_mb(matrix):
    if (
        hasattr(matrix, "data")
        and hasattr(matrix, "indices")
        and hasattr(matrix, "indptr")
    ):
        memory_bytes = matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes
    else:
        memory_bytes = matrix.nbytes

    return memory_bytes / 1024**2


def train_and_evaluate_model(
    model_name, model_type, numeric_features, categorical_features
):
    feature_columns = numeric_features + categorical_features
    X_train = train_df[feature_columns]
    y_train = train_df["sellingprice"]
    X_test = test_df[feature_columns]
    y_test = test_df["sellingprice"]

    model = build_price_model(model_type, numeric_features, categorical_features)

    train_start = time.perf_counter()
    model.fit(X_train, y_train)
    train_time = time.perf_counter() - train_start

    predict_start = time.perf_counter()
    predictions = model.predict(X_test)
    predict_time = time.perf_counter() - predict_start

    train_predictions = model.predict(X_train)
    transformed_train = model.named_steps["preprocessor"].transform(X_train)

    return {
        "Model": model_name,
        "Typ modelu": model_type,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
        "R2": r2_score(y_test, predictions),
        "MAE_train": mean_absolute_error(y_train, train_predictions),
        "RMSE_train": mean_squared_error(y_train, train_predictions) ** 0.5,
        "R2_train": r2_score(y_train, train_predictions),
        "Czas treningu (s)": train_time,
        "Czas predykcji (s)": predict_time,
        "Predykcje / s": len(X_test) / predict_time,
        "Wiersze treningowe": len(X_train),
        "Wiersze testowe": len(X_test),
        "Liczba cech po kodowaniu": transformed_train.shape[1],
        "Pamiec cech (MB)": estimate_matrix_memory_mb(transformed_train),
        "model": model,
        "predictions": np.asarray(predictions, dtype=float),
    }


model_configs = [
    ("Ridge bez MMR", "ridge", base_numeric_features),
    ("Ridge z MMR", "ridge", mmr_numeric_features),
    ("SGD bez MMR", "sgd", base_numeric_features),
    ("SGD z MMR", "sgd", mmr_numeric_features),
    ("Gradient Boosting bez MMR", "hist_gradient_boosting", base_numeric_features),
    ("Gradient Boosting z MMR", "hist_gradient_boosting", mmr_numeric_features),
]

model_results = [
    train_and_evaluate_model(
        model_name, model_type, numeric_features, categorical_features
    )
    for model_name, model_type, numeric_features in model_configs
]

results_df = pd.DataFrame(
    [
        {
            "Model": result["Model"],
            "Typ modelu": result["Typ modelu"],
            "MAE": result["MAE"],
            "RMSE": result["RMSE"],
            "R2": result["R2"],
            "Czas treningu (s)": result["Czas treningu (s)"],
            "Czas predykcji (s)": result["Czas predykcji (s)"],
            "Predykcje / s": result["Predykcje / s"],
            "Wiersze treningowe": result["Wiersze treningowe"],
            "Wiersze testowe": result["Wiersze testowe"],
            "Liczba cech po kodowaniu": result["Liczba cech po kodowaniu"],
            "Pamiec cech (MB)": result["Pamiec cech (MB)"],
        }
        for result in model_results
    ]
)

models = results_df["Model"]

quality_df = results_df[["Model", "MAE", "RMSE", "R2"]].sort_values("MAE")
print("Porownanie jakosci modeli:")
print(
    quality_df.assign(
        MAE=quality_df["MAE"].map(lambda value: f"${value:,.0f}"),
        RMSE=quality_df["RMSE"].map(lambda value: f"${value:,.0f}"),
        R2=quality_df["R2"].map(lambda value: f"{value:.3f}"),
    ).to_string(index=False)
)

best_result = min(model_results, key=lambda result: result["MAE"])
price_model = best_result["model"]
print(f"\nNajlepszy model wedlug MAE: {best_result['Model']}")

### Przedstawienie wynikow w interaktywny sposob

In [ ]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=3, subplot_titles=["MAE ($)", "RMSE ($)", "R²"])

fig.add_trace(go.Bar(
    x=models, y=results_df["MAE"], marker_color="steelblue",
    hovertemplate="<b>%{x}</b><br>MAE: $%{y:,.0f}<extra></extra>",
), row=1, col=1)

fig.add_trace(go.Bar(
    x=models, y=results_df["RMSE"], marker_color="coral",
    hovertemplate="<b>%{x}</b><br>RMSE: $%{y:,.0f}<extra></extra>",
), row=1, col=2)

fig.add_trace(go.Bar(
    x=models, y=results_df["R2"], marker_color="mediumseagreen",
    hovertemplate="<b>%{x}</b><br>R²: %{y:.3f}<extra></extra>",
), row=1, col=3)

fig.update_layout(title="Porownanie jakosci modeli", showlegend=False, height=450)
fig.update_xaxes(tickangle=30)
fig.show()

### Analiza prztrenowania.
## Kiedy niebieski slupek (trening) jest znacznie lepszy niz pomaranczowy (test), model jest przetrenowany.

In [ ]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=3, subplot_titles=["MAE ($)", "RMSE ($)", "R²"])

for col, (metric, fmt) in enumerate([
    ("MAE", "$%{y:,.0f}"),
    ("RMSE", "$%{y:,.0f}"),
    ("R2", "%{y:.3f}"),
], start=1):
    fig.add_trace(go.Bar(
        name="Trening",
        x=models,
        y=[result[f"{metric}_train"] for result in model_results],
        marker_color="steelblue",
        legendgroup="train",
        showlegend=(col == 1),
        hovertemplate=f"<b>%{{x}}</b><br>Trening: {fmt}<extra></extra>",
    ), row=1, col=col)

    fig.add_trace(go.Bar(
        name="Test", x=models, y=results_df[metric],
        marker_color="coral", legendgroup="test", showlegend=(col == 1),
        hovertemplate=f"<b>%{{x}}</b><br>Test: {fmt}<extra></extra>",
    ), row=1, col=col)

fig.update_layout(
    title="Overfitting — Trening vs Test",
    barmode="group", height=450,
    legend=dict(orientation="h", y=-0.2),
)
fig.update_xaxes(tickangle=30)
fig.show()

### Porownanie zuzycie zasobow komputera. 
## Czas treningu, czas predykcji, pamiec cech.

In [ ]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=3,
    subplot_titles=["Czas treningu (s)", "Czas predykcji (s)", "Pamiec cech (MB)"])

fig.add_trace(go.Bar(
    x=models, y=results_df["Czas treningu (s)"], marker_color="mediumpurple",
    hovertemplate="<b>%{x}</b><br>Czas treningu: %{y:.2f}s<extra></extra>",
), row=1, col=1)

fig.add_trace(go.Bar(
    x=models, y=results_df["Czas predykcji (s)"], marker_color="darkorange",
    hovertemplate="<b>%{x}</b><br>Czas predykcji: %{y:.4f}s<extra></extra>",
), row=1, col=2)

fig.add_trace(go.Bar(
    x=models, y=results_df["Pamiec cech (MB)"], marker_color="teal",
    hovertemplate="<b>%{x}</b><br>Pamiec: %{y:.1f} MB<extra></extra>",
), row=1, col=3)

fig.update_layout(title="Porownanie zasobow PC", showlegend=False, height=450)
fig.update_xaxes(tickangle=30)
fig.show()

### Wnioski
Na podstawie przeprowadzonej analizy i eksperymentów możemy wyciągnąć następujące wnioski:

**Jakość modeli:**
- Gradient Boosting osiąga zdecydowanie najlepsze wyniki zarówno pod względem MAE, RMSE jak i R². Potrafi wychwycić nieliniowe zależności, które modele liniowe (Ridge, SGD) pomijają.
- Dodanie kolumny `mmr` (szacowana cena rynkowa Manheim) dramatycznie poprawia wyniki wszystkich modeli — to najsilniejszy predyktor ceny.
- Modele liniowe (Ridge, SGD) osiągają podobne wyniki, ale SGD bywa niestabilny bez odpowiedniego doboru hiperparametrów.

**Przetrenowanie:**
- Gradient Boosting wykazuje niewielkie przetrenowanie (wyniki na zbiorze treningowym są nieco lepsze niż na testowym), ale różnica jest akceptowalna.
- Modele liniowe są bardziej odporne na przetrenowanie ze względu na regularyzację.

**Zasoby:**
- Ridge i SGD trenują się znacznie szybciej i zużywają mniej pamięci niż Gradient Boosting.
- Jeśli zależy nam na czasie (np. częste przetrenowywanie modelu na nowych danych), Ridge z MMR może być dobrym kompromisem między jakością a szybkością.

**Najważniejsze cechy wpływające na cenę:**
- Szacowana cena rynkowa (`mmr`), rok produkcji (`year`), przebieg (`odometer`) i stan techniczny (`condition`) mają największy wpływ na cenę.
- Marka, model i typ nadwozia również są istotne, ale w mniejszym stopniu.

In [ ]:
sample = model_df.sample(n=min(5000, len(model_df)), random_state=42)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=sample["odometer"],
    y=sample["sellingprice"],
    mode="markers",
    marker=dict(
        color=sample["year"],
        colorscale="Viridis",
        size=4,
        opacity=0.5,
        colorbar=dict(title="Rok produkcji"),
    ),
    hovertemplate=(
        "<b>%{customdata[0]} %{customdata[1]}</b><br>"
        "Przebieg: %{x:,.0f} mil<br>"
        "Cena: $%{y:,.0f}<br>"
        "Rok: %{marker.color}<extra></extra>"
    ),
    customdata=sample[["make", "model"]].values,
))

fig.update_layout(
    title="Cena sprzedaży vs Przebieg",
    xaxis_title="Przebieg (mile)",
    yaxis_title="Cena sprzedaży ($)",
    height=500,
)
fig.show()


In [ ]:
numeric_cols = ["sellingprice", "year", "condition", "odometer", "mmr"]
numeric_cols = [c for c in numeric_cols if c in model_df.columns]

corr = model_df[numeric_cols].corr().round(2)

fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=corr.columns.tolist(),
    y=corr.index.tolist(),
    colorscale="RdBu",
    zmid=0,
    text=corr.values.round(2),
    texttemplate="%{text}",
    hovertemplate="%{y} vs %{x}<br>Korelacja: %{z:.2f}<extra></extra>",
))

fig.update_layout(
    title="Mapa ciepła korelacji zmiennych liczbowych",
    height=450,
)
fig.show()